# Settlement — what loads pay, what generators receive, and what the TSO is left holding

A TSC-OPF yields two derivatives of the same Lagrangian, and they are different economic
objects because $P_d$ is a **parameter** and $P_g$ is a **decision variable**.

**Demand.** The envelope theorem applies directly, so

$$\pi_k \;=\; \frac{\partial\mathcal{L}}{\partial P_{d,k}} \;=\; \underbrace{\lambda_k}_{\text{energy}} \;+\; \underbrace{\mathrm{stability}_k}_{\text{dynamic}}$$

Both terms survive, because nothing forces them to cancel — nobody optimises over $P_d$. This
is Tier 1-D, already exported as `kkt_Pd_prices.csv`.

**Generation.** There is no envelope, only stationarity:

$$C'_g(P_g) \;-\; \lambda_g \;+\; \underbrace{(\bar\mu_g-\underline\mu_g)}_{\text{own limits}} \;+\; \mathrm{dyn}_g \;=\; 0$$

This is Tier 1, already exported as `Dispatch/CSV_duals/kkt_Pg_terms.csv`.

## Why the nodal price does not support the dispatch

Let a generator maximise its own profit at a posted price $\pi$, subject **only to its own
limits** — which is all it knows; it has never heard of a rotor-angle corridor:

$$\max_{P_g}\;\; \pi P_g - C_g(P_g)\qquad \text{s.t.}\quad P_g\le P_{\max}\;(\bar\mu),\quad -P_g\le -P_{\min}\;(\underline\mu)$$

Its stationarity is $\pi = C'_g + \bar\mu_g - \underline\mu_g$, which reproduces the familiar
three cases:

| where it sits | multipliers | price |
|---|---|---|
| strictly interior | $\bar\mu=\underline\mu=0$ | $\pi = C'_g$ |
| at $P_{\max}$ | $\bar\mu\ge 0$ | $\pi = C'_g+\bar\mu \ \ge\ C'_g$ — scarcity rent, the market would buy more |
| at $P_{\min}>0$ | $\underline\mu\ge 0$ | $\pi = C'_g-\underline\mu \ \le\ C'_g$ — held above its private optimum |

Subtracting that from the market's stationarity leaves

$$\lambda_g - C'_g - \bar\mu_g + \underline\mu_g \;=\; \mathrm{dyn}_g$$

The left-hand side is exactly the residual of the private problem. **It vanishes if and only
if $\mathrm{dyn}_g=0$.** Whenever a corridor binds and reaches the active-power columns, a
generator paid the uniform nodal price $\lambda_g$ would walk away from the dispatch.

The sharpest form of this: in a classical ACOPF an interior generator *always* satisfies
$\lambda_g = C'_g$. In a TSC-OPF it need not — $\bar\mu=\underline\mu=0$ and yet
$\lambda_g - C'_g = \mathrm{dyn}_g \neq 0$. That gap is not a limit rent at all; it is the
stability constraint reaching into a unit nowhere near its own capacity.

The price that restores support is one line of algebra:

$$\boxed{\;\hat\pi_g \;=\; \lambda_g - \mathrm{dyn}_g \;=\; \lambda_g + \mathrm{charge}_g\;}\qquad \mathrm{charge}_g := -\mathrm{dyn}_g$$

Substitute $\hat\pi_g$ into the private KKT and it reproduces the market dispatch by
construction. $\mathrm{charge}_g$ is the Tier 2 causer charge, which stationarity pins
per generator (`charges_by_channel.csv`, cross-checked below).

**Support here is *local*.** TSC-OPF is nonconvex; the private problem is convex, so its KKT
is sufficient *for it*, but the market's operating point is only a local optimum. A generator
could in principle prefer a distant point that no local price rules out. Say "supports the
dispatch locally" in the write-up and the claim is safe.

## What this notebook does

Runs four pricing rules against the **actual dispatch** of one finished run and reports the
money. It solves nothing and recomputes no sensitivity coefficient: every ingredient is
already on disk.

$$R(\text{load rule},\ \text{gen rule}) \;=\; \sum_k \pi^{\text{load}}_k P_{d,k} \;-\; \sum_g \pi^{\text{gen}}_g P_g$$

with **positive $R$ = TSO surplus, negative $R$ = the TSO must top up.**

**Active power only.** Nothing here settles reactive power, so $R$ is not expected to close to
zero even under uniform $\lambda$: it also carries loss, congestion and voltage rents, and the
whole unsettled reactive side. Read $R$ as a *difference between rules*, not as an absolute
revenue statement.

In [ ]:
import os, pathlib, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pd.set_option('display.width', 220)

# --- locate the run -------------------------------------------------------------------
# Same resolution as the other KKT notebooks: KKT_RUN_DIR may name a run folder directly,
# or a RESULTS root, in which case the newest suitable run wins.
NEEDED = 'Dispatch/CSV_duals'
env  = os.environ.get('KKT_RUN_DIR', '')
cand = pathlib.Path(env) if env else pathlib.Path.cwd()

def is_run(p):
    return (p / 'run_manifest.toml').is_file() and (p / NEEDED / 'kkt_Pg_terms.csv').is_file()

if is_run(cand):
    RUN = cand
else:
    runs = sorted(p for p in cand.glob('Results - *') if is_run(p))
    if not runs:
        raise SystemExit(f'No run with Dispatch/CSV_duals/kkt_Pg_terms.csv found under {cand}. '
                         'Set KKT_RUN_DIR to a run folder or a RESULTS root.')
    RUN = runs[-1]

if not (RUN / 'kkt_Pd_prices.csv').is_file():
    raise SystemExit(f'{RUN.name} has no kkt_Pd_prices.csv — the demand-side price is written by '
                     'KKT_Pd_Duals.ipynb. Run that notebook on this run first.')

rd = lambda *p: pd.read_csv(RUN.joinpath(*p), sep=';')

# --- manifest: the run describes itself -------------------------------------------------
# Flat parse, first occurrence wins — identical to the sibling notebooks. Enough for the
# scalars and the c0/c1/c2 arrays; nothing here needs the section nesting.
man = {}
for line in (RUN / 'run_manifest.toml').read_text(encoding='utf8').splitlines():
    s = line.strip()
    if '=' in s and not s.startswith('['):
        k, _, v = s.partition('=')
        man.setdefault(k.strip(), v.strip().strip('"'))

def arr(key):
    raw = man.get(key, '')
    body = raw.strip('[]').strip()
    return [float(x) for x in body.split(',')] if body else []

NETWORK   = man.get('network_form', 'KRON_REDUCED')
GENORDER  = man.get('gen_order', 'CLASSICAL_2ND')
COST_TYPE = man.get('cost_type', 'linear')
OBJ_MVA   = float(man.get('objective_MVA', 'nan'))
BOUND_PM  = man.get('bound_P_m', 'false') == 'true'

print(f'run       {RUN.name}')
print(f'path      {NETWORK} / {GENORDER}   cost_type={COST_TYPE}')
print(f'objective {OBJ_MVA:.4f} EUR/h   ({man.get("termination_status", "?")})')

In [ ]:
# --- generator side: the Tier 1 split, per unit -----------------------------------------
# Families are discovered from the `store` column, never from a hardcoded name list. That is
# deliberate: a name list is exactly what went stale once and let the bound_P_m row leak
# through a full validation pass. A new model path (AVR, governor, GFM) adds families with
# store='dyn' and flows through here untouched.
k = rd('Dispatch', 'CSV_duals', 'kkt_Pg_terms.csv')

store_of = k.drop_duplicates('family').set_index('family')['store'].to_dict()
DYN_FAM  = [f for f, s in store_of.items() if s == 'dyn']
OPF_FAM  = [f for f, s in store_of.items() if s == 'opf' and f != 'p_balance']

piv = (k.pivot_table(index=['gen_id', 'bus_id'], columns='family', values='contribution',
                     aggfunc='sum')
        .fillna(0.0).reset_index())
for f in ['p_balance', 'cost_gradient', 'RESIDUAL'] + DYN_FAM + OPF_FAM:
    if f not in piv:
        piv[f] = 0.0

G = pd.DataFrame({
    'gen':      piv.gen_id.astype(int),
    'bus':      piv.bus_id.astype(int),
    'lam':      -piv['p_balance'],           # contribution is -dual x (+1), so lambda = -contrib
    'scarcity': piv[OPF_FAM].sum(axis=1),    # (mu_upper - mu_lower), signed
    'dyn':      piv[DYN_FAM].sum(axis=1),
    'cost':     piv['cost_gradient'],
})
G['charge'] = -G.dyn
G['pi_hat'] = G.lam + G.charge

# Stationarity must close: -lambda + scarcity + dyn + C' = 0.
G['check'] = -G.lam + G.scarcity + G.dyn + G.cost
assert G.check.abs().max() < 1e-8, f'Tier 1 stationarity does not close: {G.check.abs().max():.3e}'
assert piv['RESIDUAL'].abs().max() < 1e-8, 'exported RESIDUAL row is not zero'

# Dispatched output, and the demand-side price at the generator's own bus for contrast.
gr  = rd('Dispatch', 'CSV', 'generators_report.csv')
G   = G.merge(gr[['ID', 'P_MW']].rename(columns={'ID': 'gen'}), on='gen')
pdp = rd('kkt_Pd_prices.csv')
G['pi_d_at_bus'] = G.bus.map(dict(zip(pdp.bus.astype(int), pdp.pi.astype(float))))

print(f'Generator side  [EUR/MW]   dyn families: {DYN_FAM}\n')
print(G[['gen', 'bus', 'P_MW', 'lam', 'scarcity', 'dyn', 'cost', 'charge', 'pi_hat',
         'pi_d_at_bus']].to_string(index=False, float_format=lambda x: f'{x:12.6f}'))

interior = G.scarcity.abs().max() < 1e-6
print(f'\nall units strictly interior: {interior}   (max |scarcity| = {G.scarcity.abs().max():.3e})')
if interior:
    print('  => mu = 0 and yet lambda != C\'. The whole gap is dyn: the corridor is repricing a\n'
          '     unit that is nowhere near its own capacity. This is the clean case.')

In [ ]:
# --- guards: is the split above meaningful on this run? ----------------------------------
# Warnings, not aborts. A slack corridor still gives a valid (if boring) table, and the reader
# needs to be told rather than silently handed noise.

# 1. Does any stability constraint actually bind?  `dyn` is the path-independent test and it
#    is exact: if no inequality in the dynamic block is active, stationarity in the state
#    reduces to F_x' alpha = 0, and F_x is nonsingular, so alpha = 0 and dyn = 0 identically.
dyn_max = float(G.dyn.abs().max())

#    The corridor multipliers themselves, for context. Explicit allowlist rather than a glob:
#    dual_delta.csv / dual_Delta_Omega.csv are the duals of the DEFINING EQUALITIES for the
#    state, not of the corridor, and they run three orders of magnitude larger — globbing
#    dual_delta*.csv makes this guard incapable of ever firing.
CORRIDOR_FILES = ('dual_delta_COI_lower.csv',        'dual_delta_COI_upper.csv',
                  'dual_delta_ref_lower.csv',        'dual_delta_ref_upper.csv',
                  'dual_Delta_Omega_COI_lower.csv',  'dual_Delta_Omega_COI_upper.csv',
                  'dual_Delta_Omega_abs_lower.csv',  'dual_Delta_Omega_abs_upper.csv')
gam_max, n_corr = 0.0, 0
for name in CORRIDOR_FILES:
    f = RUN / 'Transient_Stability' / 'CSV_duals' / name
    if not f.is_file():
        continue
    n_corr += 1
    num = pd.read_csv(f, sep=';').select_dtypes('number')
    if num.shape[1]:
        gam_max = max(gam_max, float(np.abs(num.to_numpy()).max()))

print(f'max |dyn|           = {dyn_max:.4g} EUR/MW')
print(f'max |corridor dual| = {gam_max:.4g}   over {n_corr} corridor export(s)')
if dyn_max < 1e-6:
    print('\n  WARNING: no stability constraint binds on this run, so every stability and dyn\n'
          '           number is numerical noise and the four settlement cells will differ only\n'
          '           by rounding. Change the corridor FORM or the horizon, not the tolerance:\n'
          '           a box on the delta variable can be satisfied for free through Q and V.')

# 2. The P_m box trap (kkt/README.md, "Pm_init is a capacity rent, not dynamics"):
#    with bound_P_m = true and P_m_source = dgen_pg_limits, the box on P_m is the SAME DGEN
#    Pmax the dispatch already imposes on P_g, so the unit's capacity rent enters the P_g row
#    twice and part of `dyn` above is really scarcity. Only scarcity + mech_pin is meaningful
#    then, and the settlement below would book a capacity rent as a stability charge.
if BOUND_PM:
    print('\n  WARNING: bound_P_m = true. If P_m_source = dgen_pg_limits, part of `dyn` above is a\n'
          '           duplicated P_g capacity rent, not dynamics. Re-solve with bound_P_m = false\n'
          '           (or a P_m box widened beyond the dispatch limits) before quoting the split.')

In [ ]:
# --- cross-check pi_hat against Tier 2, when the run carries it ---------------------------
# charges_by_channel.csv reaches the same number by a completely different route: the implicit
# function theorem on F(x,P)=0, gamma' S_P + sum_b mu_b' S_b,P = -dyn. Free validation.
ch = RUN / 'Sensitivity' / 'charges_by_channel.csv'
if ch.is_file():
    t2 = pd.read_csv(ch, sep=';')
    t2 = t2.rename(columns={t2.columns[0]: 'gen'})
    m  = G[['gen', 'charge']].merge(t2[['gen', 'charge_total']], on='gen')
    gap = (m.charge - m.charge_total).abs().max()
    print(f'Tier 1 charge vs Tier 2 charge_total: max |difference| = {gap:.3e} EUR/MW')
    # The raw gap is NOT the oracle: it carries the finite KKT residual of the solve
    # amplified by ||dx/dP||, which on a 32k-state DQ path is O(1e-5) at Ipopt tol 1e-8.
    # charges_by_channel.csv already splits that off -- read residual_unexplained.
    unex = t2['residual_unexplained'].abs().max() if 'residual_unexplained' in t2 else gap
    print(f'  unexplained by the solve own KKT residual: {unex:.3e} EUR/MW')
    assert unex < 1e-6, 'Tier 1 and Tier 2 disagree on the causer charge'
else:
    print('no Sensitivity/charges_by_channel.csv on this run — Tier 2 cross-check skipped.')

# The strongest self-test available, and it costs nothing. With a linear cost curve and every
# unit interior, pi_hat = C' exactly, so paying every generator pi_hat pays the objective.
if COST_TYPE == 'linear' and interior:
    pay_hat = float((G.P_MW * G.pi_hat).sum())
    print(f'\nsum pi_hat * P_g = {pay_hat:.4f}   vs objective_MVA = {OBJ_MVA:.4f}'
          f'   (gap {abs(pay_hat - OBJ_MVA):.3e})')
    assert abs(pay_hat - OBJ_MVA) < 1e-8 * float(G.P_MW.abs().sum()), 'pi_hat payment does not reproduce the objective'  # stationarity tol 1e-8 EUR/MW times total MW
    print('  => with linear cost and no binding own limit, pi_hat = C\' and the generators are\n'
          '     paid exactly their production cost. Zero profit. See the closing note.')

In [ ]:
# --- demand side: straight from Tier 1-D --------------------------------------------------
L = pdp[pdp.PD_MW > 0].copy()
L = L.rename(columns={'energy': 'lam'})[['bus', 'PD_MW', 'lam', 'stability', 'pi']]

print(f'Demand side  [EUR/MW]   ({len(L)} load buses, {L.PD_MW.sum():.1f} MW)\n')
print(L.to_string(index=False, float_format=lambda x: f'{x:12.6f}'))
print(f'\nlocational spread:  lambda {L.lam.max() - L.lam.min():.4f}'
      f'   ->   pi {L.pi.max() - L.pi.min():.4f} EUR/MW')

In [ ]:
# --- the 2x2 settlement -------------------------------------------------------------------
# R = what load pays - what generation is paid.   R > 0: TSO surplus.   R < 0: TSO tops up.
pay_load = {'lambda': float((L.PD_MW * L.lam).sum()),
            'pi_d':   float((L.PD_MW * L.pi ).sum())}
pay_gen  = {'lambda': float((G.P_MW  * G.lam   ).sum()),
            'pi_hat': float((G.P_MW  * G.pi_hat).sum())}

R = {(lr, gr_): pay_load[lr] - pay_gen[gr_] for lr in pay_load for gr_ in pay_gen}

print('Payments  [EUR/h]\n')
print(f'  load  @ lambda  {pay_load["lambda"]:12.2f}'
      f'      @ pi_d    {pay_load["pi_d"]:12.2f}'
      f'   ({pay_load["pi_d"] - pay_load["lambda"]:+.2f})')
print(f'  gen   @ lambda  {pay_gen["lambda"]:12.2f}'
      f'      @ pi_hat  {pay_gen["pi_hat"]:12.2f}'
      f'   ({pay_gen["pi_hat"] - pay_gen["lambda"]:+.2f})')

M = pd.DataFrame({'gen @ lambda': [R[('lambda', 'lambda')], R[('pi_d', 'lambda')]],
                  'gen @ pi_hat': [R[('lambda', 'pi_hat')], R[('pi_d', 'pi_hat')]]},
                 index=['load @ lambda', 'load @ pi_d'])
print('\nTSO residual  [EUR/h]   (+ surplus, - must top up)\n')
print(M.to_string(float_format=lambda x: f'{x:14.2f}'))

In [ ]:
# --- what the headline number is made of --------------------------------------------------
# The bottom-right cell is not a stability number; it is the conventional merchandising
# surplus PLUS two stability transfers. Separating them is what makes it readable.
R_base   = R[('lambda', 'lambda')]                     # losses + congestion + voltage rents
stab_load = float((L.PD_MW * L.stability).sum())        # extra collected from load
chg_gen   = float((G.P_MW  * G.charge   ).sum())        # extra PAID to generation (signed)
R_full   = R[('pi_d', 'pi_hat')]

assert abs((R_base + stab_load - chg_gen) - R_full) < 1e-6, 'residual decomposition does not close'

print('Residual decomposition  [EUR/h]\n')
print(f'  conventional merchandising surplus, R(lambda, lambda)   {R_base:14.2f}')
print(f'  + demand-side stability rent,  sum stability_k * P_d,k   {stab_load:+14.2f}')
print(f'  - generator causer charge,      sum charge_g * P_g       {-chg_gen:+14.2f}')
print(f'  {"=" * 56}')
print(f'  = R(pi_d, pi_hat)                                       {R_full:14.2f}')

In [ ]:
# --- profit and cost recovery, per generator ----------------------------------------------
# C_g(P) = c0 + c1 P + c2 P^2 with P in MW, matching the objective the solver minimised.
c0, c1, c2 = arr('c0'), arr('c1'), arr('c2')
gen_ids = [int(x) for x in arr('gen_ids')] or G.gen.tolist()
cost_of = {g: (c0[i] if i < len(c0) else 0.0,
               c1[i] if i < len(c1) else 0.0,
               c2[i] if i < len(c2) else 0.0) for i, g in enumerate(gen_ids)}

def total_cost(g, P):
    a0, a1, a2 = cost_of.get(g, (0.0, 0.0, 0.0))
    return a0 + a1 * P + (a2 * P * P if COST_TYPE != 'linear' else 0.0)

P = pd.DataFrame({
    'gen':          G.gen,
    'P_MW':         G.P_MW,
    'cost_EUR':     [total_cost(g, p) for g, p in zip(G.gen, G.P_MW)],
    'rev_lambda':   G.P_MW * G.lam,
    'rev_pi_hat':   G.P_MW * G.pi_hat,
})
P['profit_lambda'] = P.rev_lambda - P.cost_EUR
P['profit_pi_hat'] = P.rev_pi_hat - P.cost_EUR

print('Generator cost recovery  [EUR/h]\n')
print(P.to_string(index=False, float_format=lambda x: f'{x:12.2f}'))
print(f'\n  total profit @ lambda  {P.profit_lambda.sum():12.2f}')
print(f'  total profit @ pi_hat  {P.profit_pi_hat.sum():12.2f}')

# Make-whole: a generator cannot be left short, so any negative profit has to be topped up
# on the side. This is money the TSO owes ON TOP of the residual computed above.
uplift = {'lambda': float(np.maximum(0.0, -P.profit_lambda).sum()),
          'pi_hat': float(np.maximum(0.0, -P.profit_pi_hat).sum())}
print(f'\n  make-whole owed @ lambda  {uplift["lambda"]:12.2f}   '
      f'({int((P.profit_lambda < -1e-6).sum())} unit(s) short)')
print(f'  make-whole owed @ pi_hat  {uplift["pi_hat"]:12.2f}   '
      f'({int((P.profit_pi_hat < -1e-6).sum())} unit(s) short)')
if (P.profit_lambda < -1e-6).any():
    short = P.loc[P.profit_lambda < -1e-6, 'gen'].tolist()
    print(f'\n  Unit(s) {short} lose money at the uniform nodal price while sitting strictly\n'
          '  inside their own limits — lambda < C\' because the corridor holds them above their\n'
          '  private optimum. This is the "penalised at P_min" case with stability, not P_min,\n'
          '  doing the holding, and it is invisible to a stability-blind market.')

if P.profit_pi_hat.abs().max() < 1e-6:
    print('\n  Every unit lands on exactly zero profit. With a linear cost curve and no binding\n'
          '  own limit, marginal cost IS average cost, so pi_hat = C\' pays production cost and\n'
          '  nothing else. The entire inframarginal rent that uniform pricing gave has gone.')

In [ ]:
# --- figures ------------------------------------------------------------------------------
# Displayed only. nbconvert embeds them in the HTML report, so writing standalone .svg files
# beside it would just duplicate the same picture into the run folder — the other KKT
# notebooks never did, and this matches them.
xg = np.arange(len(G))
fig, ax = plt.subplots(figsize=(10, 4.4))
ax.bar(xg - 0.26, G.lam,    width=0.26, label=r'$\lambda_g$  (uniform nodal)', color='#4878a8')
ax.bar(xg,        G.cost,   width=0.26, label=r"$C'_g$  (marginal cost)",      color='#8c8c8c')
ax.bar(xg + 0.26, G.pi_hat, width=0.26, label=r'$\hat\pi_g$  (support price)', color='#c44e52')
ax.set_xticks(xg)
ax.set_xticklabels([f'G{g}\nbus {b}\n{p:.0f} MW' for g, b, p in zip(G.gen, G.bus, G.P_MW)])
ax.set_ylabel('EUR/MW')
ax.set_title(f'Generator prices — {NETWORK} / {GENORDER} — {RUN.name}')
ax.legend(); fig.tight_layout(); plt.show()

xl = np.arange(len(L))
fig2, ax2 = plt.subplots(figsize=(10, 3.8))
ax2.bar(xl - 0.2, L.lam,       width=0.4, label=r'$\lambda_k$', color='#4878a8')
ax2.bar(xl + 0.2, L.stability, width=0.4, label='stability',    color='#dd8452')
ax2.plot(xl, L.pi, 'ko-', ms=5, lw=1.2, label=r'$\pi_k$  total')
ax2.axhline(0, color='gray', lw=0.6)
ax2.set_xticks(xl)
ax2.set_xticklabels([f'bus {b}\n{p:.0f} MW' for b, p in zip(L.bus, L.PD_MW)])
ax2.set_ylabel('EUR/MW'); ax2.set_title('Demand prices')
ax2.legend(); fig2.tight_layout(); plt.show()

# Waterfall: how R(lambda, lambda) becomes R(pi_d, pi_hat).
steps  = ['R(lam,lam)', '+ load\nstability', '- gen\ncharge', 'R(pi_d,pi_hat)']
deltas = [R_base, stab_load, -chg_gen, None]
fig3, ax3 = plt.subplots(figsize=(9, 4.0))
run_tot = 0.0
for i, (s, d) in enumerate(zip(steps, deltas)):
    if d is None:
        ax3.bar(i, R_full, color='#4c72b0')
        ax3.text(i, R_full, f'{R_full:,.0f}', ha='center',
                 va='bottom' if R_full >= 0 else 'top', fontsize=9)
    else:
        base = 0.0 if i == 0 else run_tot
        ax3.bar(i, d, bottom=base, color='#55a868' if d >= 0 else '#c44e52')
        ax3.text(i, base + d, f'{d:+,.0f}', ha='center',
                 va='bottom' if d >= 0 else 'top', fontsize=9)
        run_tot = base + d
ax3.axhline(0, color='gray', lw=0.8)
ax3.set_xticks(range(len(steps))); ax3.set_xticklabels(steps)
ax3.set_ylabel('EUR/h'); ax3.set_title('TSO residual: where the money comes from')
fig3.tight_layout(); plt.show()

In [ ]:
# --- write the tables beside the data they describe ----------------------------------------
S = pd.DataFrame({
    'quantity': ['pay_load_lambda', 'pay_load_pi_d', 'pay_gen_lambda', 'pay_gen_pi_hat',
                 'R_lambda_lambda', 'R_lambda_pi_hat', 'R_pi_d_lambda', 'R_pi_d_pi_hat',
                 'stability_rent_load', 'causer_charge_gen', 'objective_MVA',
                 'profit_total_lambda', 'profit_total_pi_hat',
                 'makewhole_lambda', 'makewhole_pi_hat',
                 'max_abs_corridor_dual', 'max_abs_dyn'],
    'EUR_per_h': [pay_load['lambda'], pay_load['pi_d'], pay_gen['lambda'], pay_gen['pi_hat'],
                  R[('lambda', 'lambda')], R[('lambda', 'pi_hat')],
                  R[('pi_d', 'lambda')], R[('pi_d', 'pi_hat')],
                  stab_load, chg_gen, OBJ_MVA,
                  P.profit_lambda.sum(), P.profit_pi_hat.sum(),
                  uplift['lambda'], uplift['pi_hat'], gam_max, dyn_max],
})

G.to_csv(RUN / 'kkt_settlement_gen.csv',     sep=';', index=False)
L.to_csv(RUN / 'kkt_settlement_load.csv',    sep=';', index=False)
S.to_csv(RUN / 'kkt_settlement_summary.csv', sep=';', index=False)
for f in ['kkt_settlement_gen.csv', 'kkt_settlement_load.csv', 'kkt_settlement_summary.csv']:
    print(f'written: {RUN / f}')

## Reading the numbers

**$\hat\pi_g$ is a marginal signal, and paying it as a linear tariff is confiscatory.** It is
the right answer to "what price makes this unit want to do what it was told", and the wrong
answer to "what should this unit be paid". The `causer_charge_gen` line above is the whole
inframarginal rent that uniform pricing hands generation, and $\hat\pi_g$ removes exactly it.
There is no way around this by bolting on a side payment: a per-MW charge applied to every MW
*is* a change in price, and a lump-sum transfer that leaves the marginal condition alone also
leaves the incentive problem alone. Restoring incentive compatibility with a linear price and
preserving the rent are the same instrument pulled in opposite directions. So the four cells
above are a **menu**, not a ranking — pick one deliberately in the write-up and say what it
costs.

**A generator can lose money while strictly inside its own limits.** The textbook case where a
unit is underpaid is $P_g=P_{\min}$: it is held above its private optimum by its own floor, so
$\pi \le C'_g$. A TSC-OPF produces the same symptom with no floor in sight — the corridor does
the holding, $\lambda_g < C'_g$ with $\bar\mu=\underline\mu=0$, and the unit runs at a loss that
a stability-blind settlement has no vocabulary for. Read `makewhole_lambda` above: that money is
owed **on top of** the residual, and it is a stability cost misfiled as a cost-recovery problem.
The mirror image is just as real — a unit with $\mathrm{dyn}_g>0$ collects a stability premium
on top of its inframarginal rent for the same reason.

**Linear cost makes the support statement weak.** With $c_2=0$ and no binding own limit,
$\hat\pi_g = C'_g$ means profit is identically zero at *every* output in $[P_{\min},P_{\max}]$:
the generator is indifferent, not steered. The dispatch is supported only in that weak sense.
A quadratic cost curve pins $P_g^\ast$ uniquely and turns the same algebra into a genuine
selection — that is the sharper case study, and worth one extra run before the paper claims
incentive compatibility.

**$\pi_k$ at a generator's own bus is not $\hat\pi_g$.** The `pi_d_at_bus` column is printed
next to `pi_hat` precisely so the difference is visible. They are different derivatives of the
same Lagrangian: $P_d$ enters the dynamics through the ZIP load admittance, $P_g$ through
`Pe_init`, `Pm_init` and the first swing step. Reporting one as the other is a category error
that the two columns make hard to commit by accident.

**The residual is not all stability.** $R(\lambda,\lambda)$ is already non-zero before any
stability term appears — losses, congestion and voltage rents, plus a reactive side this
notebook does not settle at all. Stability *moves* the residual; it does not create it. Quote
the decomposition, never the bottom line alone.

**The corridor has to bind, and the duals have to be converged.** If `max_abs_corridor_dual`
is ~0 the guard above has already said so and every number here is noise. Even when it binds,
the stability terms are sums over transient duals, so a run terminating
`ALMOST_LOCALLY_SOLVED` degrades them while $\lambda$ stays exact — use Ipopt with
`hessian_approximation = "exact"` and `tol = 1e-8`, which on these cases converged better
*and* roughly three times faster than L-BFGS.